# Building the machine-learning dataset

## What do we want to predict, and from what?

The **target** is the sea-ice velocity vector, `siu` and `siv` (eastward and northward components, in m/s).
The **inputs** have to be things that are known everywhere, otherwise the trained model could not be applied to the
whole Arctic. Physics tells us what matters: the momentum balance of sea ice says that on daily time scales ice
motion is driven mostly by the wind stress, with the ocean, the Coriolis force, the sea-surface tilt and internal ice
stresses (which depend on how much ice there is) doing the rest.

```{figure} images/slide_momentum_balance.png
:width: 85%
:name: momentum

The momentum balance of sea ice. The wind alone explains up to about 70 % of the variance of ice motion on short time scales (Thorndike & Colony 1982).
```

So our inputs are the **wind** (`uas`, `vas`), the **ice concentration** (`sic`, a proxy for how much the ice resists
being pushed), the **day of year** (season) and the **position** (latitude and longitude — the model can learn that
the ice behaves differently in the Beaufort Gyre than near Fram Strait). Longitude goes in as its cosine and sine so
that 179 °E and 179 °W are neighbours, not opposites. Seven numbers in, two numbers out:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from oaisis import data, plots, ml, metrics, selectors, experiments, teaching

In [ ]:
teaching.black_box();
print("inputs  X:", data.FEATURES)
print("outputs Y:", data.TARGETS)

## Learning as a black box

A machine-learning model is a function with adjustable **parameters** that turns `X` into a prediction of `Y`.
*Training* means choosing the parameters so that the predictions match the data. The simplest such model is
**linear regression**, `Y = W X + B`: a weight `W` for each input and a bias `B`. A neural network is the same idea
with many more parameters arranged in layers, and we will come to it in two pages. Whatever the model, the
data have to be arranged the same way first: one row per sample, one column per input.

In [ ]:
X, Y = data.obs_table(members=["r1i1p1f1"])      # one member, all 11 242 ITP samples
print("X:", X.shape, " Y:", Y.shape)
print("first sample:  X =", np.round(X[0], 2), "  Y =", np.round(Y[0], 3))

## Three sets: training, validation, test

A model that is only ever checked on the data it was trained on can look good while having learned nothing useful —
it may just have *memorised* the samples. So the data are split into three parts that never mix:

| set | used for |
|---|---|
| **training** | fitting the parameters (the model *learns* from these) |
| **validation** | watching the model during training on data it does not learn from, and choosing hyperparameters |
| **test** | one final, honest score on data the model has never seen |

Here the natural way to split is **by ensemble member**: the 31 members share the same ITP positions and dates but
have independent weather, so a member is a genuinely new realisation. Splitting the *samples* at random instead would
put consecutive days of the same track in both training and test — and consecutive days are nearly identical, so the
test score would be flattered. Member `r1` goes into the test set because the full-Arctic maps we reconstruct later
come from it.

In [ ]:
teaching.split_diagram()
for name, members in data.SPLIT.items():
    print(f"{name:5s}: {len(members):2d} members  ({members[0]} ... {members[-1]})")

## Feature scaling

Wind is tens of m/s, concentration runs from 0 to 100 %, day of year from 1 to 366, cos/sin of longitude from −1 to 1.
A model that mixes these as they are would have to find weights spanning several orders of magnitude, and the training
algorithm (gradient descent, next pages) takes steps of a size that is the same for every parameter — it struggles
badly when the inputs live on different scales. So we **scale every input to order one**:

* `uas`, `vas`: *standardize* — subtract the training-set mean, divide by the training-set standard deviation;
* `sic`, `doy`, `lat`: divide by a fixed constant (100, 365, 90) so that the scaled value keeps its meaning;
* `cos_lon`, `sin_lon`: already in [−1, 1].

The targets are standardized too, so that the loss is a dimensionless number. The statistics come from the **training
set only** — using the test set here would already leak information about it. `data.prepared()` does the split and the
scaling in one go:

In [ ]:
p = data.prepared()          # X_tr, Y_tr, X_va, ... in physical units; x_tr, y_tr, ... scaled; p.scaler to undo it
teaching.feature_ranges(p);

```{admonition} Try it
:class: tip
`data.Scaler.fixed` holds the fixed divisors. In *Live Code* mode set `data.Scaler.fixed = {}` before calling
`data.prepared()` and look at the ranges: now every input is standardized. Either choice works for the network; the
fixed scaling just keeps `sic = 0.5` meaning "50 %".
```

Next: [the simplest model — a straight line through our data](03_linear_regression.ipynb).